In [1]:
import sys
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaLLM
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_core.messages import HumanMessage, AIMessage


d:\github\Learning\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Administrator\AppData\Local\Temp\ipykernel_6172\1301479741.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [2]:
# ==========================================
# 1. LLM Setup
# ==========================================
print("🔄 Initializing Ollama (Llama 3.1)...")
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
    timeout=60  
)
#connect cromadb vector db
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  # embedding তৈরির সময় যেটা ব্যবহার করেছিলে
)

vector_db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

🔄 Initializing Ollama (Llama 3.1)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6829.60it/s]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_6172\3262766231.py:16: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


In [4]:
# Initialize chat history as a list of message objects
chat_history = []

def format_chat_history(messages):
    """Format chat history into a readable string."""
    if not messages:
        return ""
    formatted = []
    for msg in messages:
        if isinstance(msg, HumanMessage):
            formatted.append(f"Human: {msg.content}")
        elif isinstance(msg, AIMessage):
            formatted.append(f"AI: {msg.content}")
    return "\n".join(formatted)

# Updated template with proper history formatting
template = """You are a helpful assistant answering questions based strictly on the provided context.
If you don't know the answer or if it's not in the context, say that you don't know.
Do not make things up.

Previous conversation:
{chat_history}

Context:
{context}

Question: 
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

def run_rag_pipeline(user_question):
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})
    
    # Create the RAG chain with chat history
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
            "chat_history": lambda _: format_chat_history(chat_history)
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    
    print(f"\nQuerying: '{user_question}'\n")
    response = rag_chain.invoke(user_question)
    
    # Update history with proper message objects
    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=response))
    
    print("---History---")
    print(chat_history)

    print("--- AI Answer ---")
    print(response)
    print()

In [5]:
query = "who is abu sufiun?"
run_rag_pipeline(query)


Querying: 'who is abu sufiun?'

---History---
[HumanMessage(content='who is abu sufiun?', additional_kwargs={}, response_metadata={}), AIMessage(content='Abu Sufiun is a Full Stack Developer with expertise in Laravel & React.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
--- AI Answer ---
Abu Sufiun is a Full Stack Developer with expertise in Laravel & React.



In [6]:
run_rag_pipeline("how many skills he has?")


Querying: 'how many skills he has?'

---History---
[HumanMessage(content='who is abu sufiun?', additional_kwargs={}, response_metadata={}), AIMessage(content='Abu Sufiun is a Full Stack Developer with expertise in Laravel & React.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='how many skills he has?', additional_kwargs={}, response_metadata={}), AIMessage(content='According to the context, Abu Sufiun has 12 skills listed:\n\n1. RESTful API\n2. Laravel\n3. React\n4. Bootstrap\n5. Tailwind CSS\n6. MS-SQL\n7. Oracle\n8. My-SQL\n9. OOP (Object-Oriented Programming)\n10. PHP\n11. JavaScript\n12. C\n13. AI (Artificial Intelligence)\n14. ML (Machine Learning)\n\nSo, he has 14 skills in total.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
--- AI Answer ---
According to the context, Abu Sufiun has 12 skills listed:

1. RESTful API
2. Laravel
3. React
4. Bootstrap
5. Tailwind CSS
6. MS-SQL
7. Ora

In [7]:
run_rag_pipeline("did he need to learn any thing else?")


Querying: 'did he need to learn any thing else?'

---History---
[HumanMessage(content='who is abu sufiun?', additional_kwargs={}, response_metadata={}), AIMessage(content='Abu Sufiun is a Full Stack Developer with expertise in Laravel & React.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='how many skills he has?', additional_kwargs={}, response_metadata={}), AIMessage(content='According to the context, Abu Sufiun has 12 skills listed:\n\n1. RESTful API\n2. Laravel\n3. React\n4. Bootstrap\n5. Tailwind CSS\n6. MS-SQL\n7. Oracle\n8. My-SQL\n9. OOP (Object-Oriented Programming)\n10. PHP\n11. JavaScript\n12. C\n13. AI (Artificial Intelligence)\n14. ML (Machine Learning)\n\nSo, he has 14 skills in total.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='did he need to learn any thing else?', additional_kwargs={}, response_metadata={}), AIMessage(content="Based on the provide